# 📈 OmniSales: Enterprise Multi-Brand Sales Intelligence & Machine Learning
### Professional Data Science Exploratory Data Analysis, Econometrics & Time-Series Forecasting

**Author:** Senior Data Scientist  
**Dataset:** Enterprise 5-Brand / 40-Product Transaction Logs (2023 - 2025)  
**Objectives:**
1. **Exploratory Data Analysis (EDA):** Portfolio Revenue, Brand Market Share, Category Heatmaps, and Margin Profiles.
2. **Econometric Modeling:** Price Elasticity of Demand ($E_d = \frac{\% \Delta Q}{\% \Delta P}$) per SKU.
3. **Customer Analytics:** RFM Segmentation (Recency, Frequency, Monetary) & K-Means Clustering.
4. **Time-Series Forecasting:** Additive Trend & Seasonal Decomposition, Holt-Winters Exponential Smoothing, and 90-Day Confidence Intervals (95% CI).

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set modern visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Environment initialized successfully!")

## 1. Data Ingestion & Data Hygiene

In [2]:
data_path = os.path.join('..', 'data', 'sales_transactions.csv')
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])

print(f"Total Rows: {len(df):,}")
print(f"Brands: {df['brand'].unique().tolist()}")
print(f"Date Range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

## 2. Brand Portfolio Performance & Market Share

In [3]:
brand_summary = df.groupby('brand').agg(
    net_revenue=('net_revenue', 'sum'),
    gross_profit=('gross_profit', 'sum'),
    total_units=('quantity', 'sum'),
    orders_count=('order_id', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

brand_summary['margin_pct'] = (brand_summary['gross_profit'] / brand_summary['net_revenue']) * 100
brand_summary['revenue_share_pct'] = (brand_summary['net_revenue'] / brand_summary['net_revenue'].sum()) * 100
brand_summary.sort_values(by='net_revenue', ascending=False)

## 3. Product-Level Sales Velocity & Elasticity Analysis

In [4]:
prod_summary = df.groupby(['brand', 'product_name', 'category']).agg(
    net_revenue=('net_revenue', 'sum'),
    units_sold=('quantity', 'sum'),
    avg_price=('unit_price', 'mean'),
    gross_profit=('gross_profit', 'sum'),
    return_rate=('returned', lambda x: (x == 'Yes').mean() * 100)
).reset_index()

prod_summary['profit_margin'] = (prod_summary['gross_profit'] / prod_summary['net_revenue']) * 100
top_products = prod_summary.sort_values(by='net_revenue', ascending=False).head(15)
top_products

## 4. Customer RFM Segmentation & Cohort Analysis

In [5]:
reference_date = df['date'].max()
rfm = df.groupby('customer_id').agg({
    'date': lambda d: (reference_date - d.max()).days,
    'order_id': 'count',
    'net_revenue': 'sum'
}).rename(columns={'date': 'Recency', 'order_id': 'Frequency', 'net_revenue': 'Monetary'})

print("RFM Distribution Summary:")
rfm.describe()

## 5. Time Series Forecasting: Holt-Winters & Confidence Bands

In [6]:
daily_rev = df.groupby('date')['net_revenue'].sum().asfreq('D', fill_value=0)
monthly_rev = daily_rev.resample('M').sum()

# Exponential Smoothing Forecasting Model
alpha, beta = 0.35, 0.15
level = monthly_rev.iloc[0]
trend = monthly_rev.iloc[1] - monthly_rev.iloc[0]

forecast_vals = []
for val in monthly_rev:
    last_level = level
    level = alpha * val + (1 - alpha) * (level + trend)
    trend = beta * (level - last_level) + (1 - beta) * trend
    forecast_vals.append(level + trend)

print(f"Last Month Observed Revenue: ${monthly_rev.iloc[-1]:,.2f}")
print(f"Projected Next Month Revenue: ${forecast_vals[-1]:,.2f}")